In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import pandas as pd

df = pd.read_csv(
    r'C:\Users\Dell\Documents\NLP For ML\Emotions detection\Emotions dataset for NLP\train.txt',
    sep=';',
    header=None,
    names=['text', 'emotion']
)
val_df = pd.read_csv(
    r"C:\Users\Dell\Documents\NLP For ML\Emotions detection\Emotions dataset for NLP\val.txt",
    sep=';',
    header=None,
    names=['text', 'emotion']
)
test_df = pd.read_csv(
    r"C:\Users\Dell\Documents\NLP For ML\Emotions detection\Emotions dataset for NLP\test.txt",
    sep=';',
    header=None,
    names=['text', 'emotion']
)

In [3]:
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [4]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [5]:
unique_emotions=df['emotion'].unique()

In [6]:
unique_emotions

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [7]:
emotion_numbers={}

In [8]:
i=0
for emo in unique_emotions:
    emotion_numbers[emo]=i
    i +=1

In [9]:
emotion_numbers

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}

In [10]:
df['emotion']=df['emotion'].map(emotion_numbers)

In [11]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


Text Cleaning

In [12]:
df['text']=df['text'].apply(lambda x: x.lower()) 

Remove Punctuation

In [13]:
import string
def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))
 


In [14]:
df['text']=df['text'].apply(remove_punc)

Remove Numbers

In [15]:
def remove_numbers(txt):
    new=""
    for i in txt:
        if not i.isdigit():
            new=new +i
    return new

In [16]:
df['text']=df['text'].apply(remove_numbers)

Remove Emojies and special characters

In [17]:
def remove_emojis(txt):
    new=""
    for i in txt:
        if i.isascii():
            new +=i
    return new

In [18]:
df['text']=df['text'].apply(remove_emojis)

Remove Stopwords

In [19]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [20]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [21]:
stop_words=set(stopwords.words('english'))

In [22]:
len(stop_words)

198

In [23]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [24]:
def remove(txt):
    words=txt.split()
    cleaned=[]
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)

In [25]:
df['text']=df['text'].apply(remove)

In [26]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [27]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [28]:

val_df['emotion']  = val_df['emotion'].map(emotion_numbers)
test_df['emotion'] = test_df['emotion'].map(emotion_numbers)

X_train = df['text']
y_train = df['emotion']

X_val = val_df['text']
y_val = val_df['emotion']

X_test = test_df['text']
y_test = test_df['emotion']

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [30]:
bow_vectorizer=CountVectorizer()

In [31]:
X_train_bow=bow_vectorizer.fit_transform(X_train)
X_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 144778 stored elements and shape (16000, 15046)>

In [32]:
X_test_bow = bow_vectorizer.transform(X_test)

In [33]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [34]:
nb_model=MultinomialNB()

In [35]:
nb_model.fit(X_train_bow,y_train)

MultinomialNB()

Testing

In [36]:
X_val_bow = bow_vectorizer.transform(X_val)

val_pred = nb_model.predict(X_val_bow)
val_pred

array([0, 0, 5, ..., 5, 5, 5])

In [37]:
from sklearn.metrics import accuracy_score

print("Validation Accuracy:",
      accuracy_score(y_val, val_pred))

Validation Accuracy: 0.788


In [38]:
X_test_bow = bow_vectorizer.transform(X_test)

test_pred = nb_model.predict(X_test_bow)

In [39]:
print("Test Accuracy:",
      accuracy_score(y_test, test_pred))

Test Accuracy: 0.7895


TFIDF Vectorizer

In [40]:
tfidf = TfidfVectorizer()

In [41]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

In [42]:
nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [43]:
val_pred = nb2_model.predict(X_val_tfidf)

from sklearn.metrics import accuracy_score
print("Validation Accuracy:", accuracy_score(y_val, val_pred))

Validation Accuracy: 0.6775


In [44]:
test_pred = nb2_model.predict(X_test_tfidf)

print("Test Accuracy:", accuracy_score(y_test, test_pred))

Test Accuracy: 0.684


Logistic Regression Model

In [45]:
from sklearn.linear_model import LogisticRegression

In [46]:
logistic_model=LogisticRegression(max_iter=1000)

In [47]:
logistic_model.fit(X_train_bow,y_train)

LogisticRegression(max_iter=1000)

In [48]:
X_val_bow = bow_vectorizer.transform(X_val)

val_pred = logistic_model.predict(X_val_bow)
val_pred

array([0, 0, 5, ..., 5, 5, 5])

In [49]:
from sklearn.metrics import accuracy_score

print("Validation Accuracy:",
      accuracy_score(y_val, val_pred))

Validation Accuracy: 0.897


In [50]:
X_test_bow = bow_vectorizer.transform(X_test)

test_pred = logistic_model.predict(X_test_bow)

In [51]:
print("Test Accuracy:",
      accuracy_score(y_test, test_pred))

Test Accuracy: 0.89


Logistic Regression for TFIDF

In [52]:
logistic_model2=LogisticRegression(max_iter=1000)
logistic_model2.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [53]:
val_pred = logistic_model2.predict(X_val_tfidf)

from sklearn.metrics import accuracy_score
print("Validation Accuracy:", accuracy_score(y_val, val_pred))

Validation Accuracy: 0.873


In [54]:
test_pred = logistic_model2.predict(X_test_tfidf)

print("Test Accuracy:", accuracy_score(y_test, test_pred))

Test Accuracy: 0.865


In [55]:
import joblib
joblib.dump(logistic_model, "logistic_model.pkl")
joblib.dump(bow_vectorizer, "bow_vectorizer.pkl")

['bow_vectorizer.pkl']